In [33]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
# Step 2: Load Dataset
column_names = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
                'acceleration', 'model_year', 'origin', 'car_name']
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'

df = pd.read_csv(url, delim_whitespace=True, names=column_names, na_values='?')

In [35]:
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    float64
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    int64  
 8   car_name      398 non-null    object 
dtypes: float64(5), int64(3), object(1)
memory usage: 28.1+ KB


In [37]:
# Step 3: Select continuous features: only feature with float data types are considered
continuous_cols = ['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']
df_cont = df[continuous_cols]

In [38]:
df_cont.head()

,mpg,displacement,horsepower,weight,acceleration
0,18.0,307.0,130.0,3504.0,12.0
1,15.0,350.0,165.0,3693.0,11.5
2,18.0,318.0,150.0,3436.0,11.0
3,16.0,304.0,150.0,3433.0,12.0
4,17.0,302.0,140.0,3449.0,10.5


In [39]:
# Step 4: Impute missing values with mean
imputer = SimpleImputer(strategy='mean')
df_cont_imputed = pd.DataFrame(imputer.fit_transform(df_cont), columns=continuous_cols)

In [40]:
# Step 5: Standardize data (recommended for clustering)
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cont_imputed)

In [41]:
# Hierarchical Clustering with optimal K (UPDATED: Added metric/affinity='euclidean')
optimal_hierarchical_k = 3  
agglo = AgglomerativeClustering(
    n_clusters=optimal_hierarchical_k,
    metric='euclidean',  
    linkage='average'
)
agglo_labels = agglo.fit_predict(df_scaled)
agglo_labels[:10]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=int64)

In [42]:
# Step 9: Add results to DataFrame
df['hierarchical_cluster'] = agglo_labels
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name,hierarchical_cluster
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu,1
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320,1
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite,1
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst,1
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino,1


In [43]:
# Step 10: Cluster statistics
print("\nHierarchical Cluster Stats:")
print(df.groupby('hierarchical_cluster')[continuous_cols].agg(['mean', 'var']))

print("\nOrigin Class Stats:")
print(df.groupby('origin')[continuous_cols].agg(['mean', 'var']))


Hierarchical Cluster Stats:
                            mpg            displacement               \
                           mean        var         mean          var   
hierarchical_cluster                                                   
0                     26.177441  41.303375   144.304714  3511.485383   
1                     14.528866   4.771033   348.020619  2089.499570   
2                     43.700000   0.300000    91.750000    12.250000   

                      horsepower                   weight                 \
                            mean         var         mean            var   
hierarchical_cluster                                                       
0                      86.120275  294.554450  2598.414141  299118.709664   
1                     161.804124  674.075816  4143.969072  193847.051117   
2                      49.000000    4.000000  2133.750000   21672.916667   

                     acceleration            
                             mean  

In [44]:
print("\nHierarchical vs Origin:\n", pd.crosstab(df['origin'], df['hierarchical_cluster']))


Hierarchical vs Origin:
 hierarchical_cluster    0   1  2
origin                          
1                     152  97  0
2                      66   0  4
3                      79   0  0
